# MIMIC-IV Data Processing

This notebook demonstrates how to process MIMIC-IV patient data using pandas.
We will load a sample of patients, extract specific columns, and save the result to a Parquet file.

In [2]:
import pandas as pd

# Define the number of patients to process
# This constant limits the data loading to optimize memory usage
NUM_PATIENTS = 1000

# Specify the path to your MIMIC-IV patients file
# Update this path to point to your local MIMIC-IV data directory
# The file can be either 'patients.csv' or 'patients.csv.gz' (compressed)
patients_file_path = 'C:\\Users\\Eli\\Data\\physionet.org\\files\\mimiciv\\3.1\\hosp\\patients.csv.gz'

# Read the MIMIC-IV patients file
# nrows parameter limits the number of rows loaded to NUM_PATIENTS
# usecols parameter extracts only the specified columns to reduce memory footprint
patients_df = pd.read_csv(
    patients_file_path,
    nrows=NUM_PATIENTS,  # Load only the first 1000 rows
    usecols=['subject_id', 'gender', 'anchor_age']  # Extract only required columns
)

# Display basic information about the loaded data
print(f"Loaded {len(patients_df)} patient records")
print(f"\nDataFrame shape: {patients_df.shape}")
print(f"\nColumn names: {list(patients_df.columns)}")
print(f"\nFirst few rows:")
print(patients_df.head())

# Save the extracted DataFrame to a Parquet file
# Parquet format provides efficient compression and fast read/write operations
patients_df.to_parquet('patients_sample.parquet', index=False)
print(f"\nData successfully saved to 'patients_sample.parquet'")

Loaded 1000 patient records

DataFrame shape: (1000, 3)

Column names: ['subject_id', 'gender', 'anchor_age']

First few rows:
   subject_id gender  anchor_age
0    10000032      F          52
1    10000048      F          23
2    10000058      F          33
3    10000068      F          19
4    10000084      M          72

Data successfully saved to 'patients_sample.parquet'


In [ ]:
# Step 1: Load the dataframe from the previous step
patients_df = pd.read_parquet('patients_sample.parquet')

# Step 2: Create a mapping dictionary to convert gender values
gender_mapping = {'M': 'Male', 'F': 'Female'}

# Step 3 & 4: Create the patient_text column with the specified format
# Map gender values and format the text as "Age {anchor_age} {gender_mapped}"
patients_df['patient_text'] = patients_df.apply(
    lambda row: f"Age {row['anchor_age']} {gender_mapping[row['gender']]}",
    axis=1
)

# Step 5: Create the final dataframe with only subject_id and patient_text
patients_text_df = patients_df[['subject_id', 'patient_text']]

# Step 6: Display the first 5 records to verify the format
print("First 5 records of the processed data:")
print(patients_text_df.head())

# Step 7: Save the processed dataframe to a new parquet file
patients_text_df.to_parquet('patients_text_representation.parquet', index=False)
print(f"\nData successfully saved to 'patients_text_representation.parquet'")
print(f"Total records processed: {len(patients_text_df)}")

In [ ]:
# Step 8: Generate Patient Embeddings using Clinical_ModernBERT

# Import required libraries
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

# Check for GPU availability and set device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS (Metal Performance Shaders)")
else:
    device = torch.device("cpu")
    print("Using CPU")

# Load tokenizer and model from Hugging Face
model_name = "Simonlee711/Clinical_ModernBERT"
print(f"
Loading tokenizer and model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, add_pooling_layer=False).to(device)
model.eval()  # Set model to evaluation mode
print("Model loaded successfully")

# Load the patient text data from previous step
print("
Loading patient text data...")
patients_text_df = pd.read_parquet("patients_text_representation.parquet")
print(f"Loaded {len(patients_text_df)} patient records")
print(f"Columns: {list(patients_text_df.columns)}")

# Define the embedding function with batch processing and mean pooling
def get_embeddings(text_list, batch_size=16):
    """
    Generate embeddings for a list of text inputs using Clinical_ModernBERT.
    
    Args:
        text_list: List of text strings to encode
        batch_size: Number of texts to process in each batch (default: 16)
    
    Returns:
        numpy array of embeddings with shape (num_texts, embedding_dim)
    """
    all_embeddings = []
    
    # Process in batches to avoid OOM errors
    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i + batch_size]
        
        # Tokenize with padding, truncation, and extended context window
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=8192,  # ModernBERT extended context window
            return_tensors="pt"
        ).to(device)
        
        # Generate embeddings without gradient computation
        with torch.no_grad():
            outputs = model(**inputs)
            
            # Extract last hidden state
            last_hidden_state = outputs.last_hidden_state
            
            # Apply attention mask-aware mean pooling
            # This properly excludes padding tokens from the average for more precise embeddings
            attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
            sum_embeddings = torch.sum(last_hidden_state * attention_mask, 1)
            sum_mask = attention_mask.sum(1)
            sum_mask = torch.clamp(sum_mask, min=1e-9)  # Avoid division by zero
            embeddings = sum_embeddings / sum_mask
            
            # Move to CPU and convert to numpy
            embeddings = embeddings.cpu().numpy()
            all_embeddings.append(embeddings)
    
    # Concatenate all batch embeddings
    return np.vstack(all_embeddings)

# Apply the embedding function to the patient_text column
print("
Generating embeddings...")
patient_texts = patients_text_df["patient_text"].tolist()
embeddings = get_embeddings(patient_texts, batch_size=16)
print(f"Generated {len(embeddings)} embeddings")

# Create output DataFrame with subject_id and embedding
# This keeps the primary key aligned with the embedding vectors
embeddings_df = pd.DataFrame({
    "subject_id": patients_text_df["subject_id"].values,
    "embedding": list(embeddings)  # Convert array rows to list of arrays
})

# Save the embeddings to parquet file
output_file = "patient_clinical_modernbert_embeddings.parquet"
embeddings_df.to_parquet(output_file, index=False)
print(f"
Embeddings saved to '{output_file}'")

# Print verification information
print(f"
Final DataFrame shape: {embeddings_df.shape}")
print(f"Embedding dimension: {embeddings[0].shape[0]}")
print(f"
First few rows:")
print(embeddings_df.head())